In [3]:
import pandas as pd
import numpy as np
import random
import copy
import os
import json
import time

# ==========================================
# PARAMETER DPSO & CONSTRAINT VRP
# ==========================================
SWARM_SIZE = 50
MAX_ITER = 300
PATIENCE = 80

P_COG = 0.8
P_SOC = 0.9
P_INERTIA = 0.5

# Constraint Waktu (dalam satuan menit)
SERVICE_TIME = 20          # 20 menit service hour per puskesmas
WORK_HOUR_LIMIT = 600      # 10 jam kerja = 600 menit

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("✅ Libraries and parameters loaded successfully!")

✅ Libraries and parameters loaded successfully!


In [4]:
def clean_coordinate(val, is_lat=True):
    """Pembersih otomatis koordinat rusak akibat format regional CSV Excel"""
    val_str = str(val).strip().replace('"', '').replace("'", "")
    digits = val_str.replace(',', '').replace('.', '').replace('-', '')
    if not digits:
        return 0.0
    
    is_negative = val_str.startswith('-')
    
    if is_lat:
        # Latitude Surabaya pasti diawali -7.xxxx
        cleaned = f"-{digits[0]}.{digits[1:]}" if is_negative else f"{digits[0]}.{digits[1:]}"
    else:
        # Longitude Surabaya pasti diawali 112.xxxx
        cleaned = f"{digits[:3]}.{digits[3:]}"
    return float(cleaned)

def get_priority_score(jaringan, jenis):
    """
    Mapping prioritas sesuai dataset:
    1: Induk - Rawat Inap (Tertinggi)
    2: Induk - Rawat Jalan
    3: Pembantu - Rawat Jalan (Terendah)
    """
    jaringan_lower = str(jaringan).lower()
    jenis_lower = str(jenis).lower()
    
    if "induk" in jaringan_lower and "inap" in jenis_lower:
        return 1
    elif "induk" in jaringan_lower and "jalan" in jenis_lower:
        return 2
    else:
        return 3 # Default terendah untuk pembantu / pusk keliling
    
print("✅ Coordinate cleaner and priority function defined!")

✅ Coordinate cleaner and priority function defined!


In [5]:
def decode_and_evaluate(permutation, dist_matrix, time_matrix, lokasi_list, priority_map):
    routes = []
    current_route = []
    current_time = 0.0
    current_dist = 0.0
    
    # Indeks 0 = Depot Utama (UPTD Gudang Farmasi)
    for node in permutation:
        if not current_route:
            time_needed = time_matrix[0][node] + SERVICE_TIME + time_matrix[node][0]
            current_route.append(node)
            current_time = time_matrix[0][node] + SERVICE_TIME
            current_dist = dist_matrix[0][node]
        else:
            last_node = current_route[-1]
            time_to_next = time_matrix[last_node][node] + SERVICE_TIME
            time_return_from_next = time_matrix[node][0]
            
            # Cek limit jam kerja kurir (10 jam)
            if current_time + time_to_next + time_return_from_next <= WORK_HOUR_LIMIT:
                current_route.append(node)
                current_time += time_to_next
                current_dist += dist_matrix[last_node][node]
            else:
                # Kurir pulang ke depot
                current_time += time_matrix[last_node][0]
                current_dist += dist_matrix[last_node][0]
                routes.append({"nodes": current_route, "waktu": current_time, "jarak": current_dist})
                
                # Oper ke kurir baru
                current_route = [node]
                current_time = time_matrix[0][node] + SERVICE_TIME
                current_dist = dist_matrix[0][node]
                
    if current_route:
        current_time += time_matrix[current_route[-1]][0]
        current_dist += dist_matrix[current_route[-1]][0]
        routes.append({"nodes": current_route, "waktu": current_time, "jarak": current_dist})
        
    total_actual_distance = sum(r["jarak"] for r in routes)
    total_actual_time = sum(r["waktu"] for r in routes)
    
    # Perhitungan Penalty Prioritas Kunjungan
    priority_penalty = 0.0
    for r in routes:
        for i in range(len(r["nodes"]) - 1):
            p1 = priority_map.get(lokasi_list[r["nodes"][i]], 3)
            p2 = priority_map.get(lokasi_list[r["nodes"][i+1]], 3)
            if p1 > p2:  # Jika melanggar urutan prioritas
                priority_penalty += 500.0
                
    fitness_value = total_actual_distance + priority_penalty
    return fitness_value, routes, total_actual_distance, total_actual_time

print("✅ Decoder and fitness evaluation function updated successfully!")

✅ Decoder and fitness evaluation function updated successfully!


In [6]:
def get_swap_sequence(target, current):
    seq = []
    temp = current.copy()
    for i in range(len(target)):
        if temp[i] != target[i]:
            j = temp.index(target[i])
            seq.append((i, j))
            temp[i], temp[j] = temp[j], temp[i]
    return seq

def apply_velocity(route, velocity):
    new_route = route.copy()
    for i, j in velocity:
        new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route

print("✅ Swap operators compiled successfully!")

✅ Swap operators compiled successfully!


In [20]:
def run_dpso_vrp(dist_file, time_file, coord_df, priority_map):
    df_dist = pd.read_csv(dist_file, index_col=0)
    df_time = pd.read_csv(time_file, index_col=0)
    
    lokasi_list = [str(col).strip() for col in df_dist.columns]
    dist_matrix = df_dist.values
    time_matrix = df_time.values
    
    nodes = list(range(1, len(lokasi_list)))
    if len(nodes) == 0: return {}
        
    swarm = [random.sample(nodes, len(nodes)) for _ in range(SWARM_SIZE)]
    velocities = [[] for _ in range(SWARM_SIZE)]
    
    pbest = copy.deepcopy(swarm)
    pbest_fit = []
    for p in swarm:
        fit, _, _, _ = decode_and_evaluate(p, dist_matrix, time_matrix, lokasi_list, priority_map)
        pbest_fit.append(fit)
        
    best_idx = int(np.argmin(pbest_fit))
    gbest = copy.deepcopy(pbest[best_idx])
    gbest_fit, gbest_routes, gbest_dist, gbest_time = decode_and_evaluate(gbest, dist_matrix, time_matrix, lokasi_list, priority_map)
    
    vmax = int(1.5 * len(nodes))
    no_improve = 0
    
    # 🌟 STEP 1: Inisialisasi list untuk mencatat riwayat konvergensi
    riwayat_fitness = []
    
    for _ in range(MAX_ITER):
        improved = False
        for i in range(SWARM_SIZE):
            fit, _, _, _ = decode_and_evaluate(swarm[i], dist_matrix, time_matrix, lokasi_list, priority_map)
            if fit < pbest_fit[i]:
                pbest[i] = copy.deepcopy(swarm[i])
                pbest_fit[i] = fit
                
        best_idx = int(np.argmin(pbest_fit))
        if pbest_fit[best_idx] < gbest_fit:
            gbest = copy.deepcopy(pbest[best_idx])
            gbest_fit, gbest_routes, gbest_dist, gbest_time = decode_and_evaluate(gbest, dist_matrix, time_matrix, lokasi_list, priority_map)
            improved = True
            
        # 🌟 STEP 2: Catat nilai gbest_fit terbaik pada iterasi saat ini
        riwayat_fitness.append(round(gbest_fit, 2))
            
        if improved: no_improve = 0
        else: no_improve += 1
            
        if no_improve >= PATIENCE: break
            
        for i in range(SWARM_SIZE):
            v_cog = get_swap_sequence(pbest[i], swarm[i])
            v_soc = get_swap_sequence(gbest, swarm[i])
            new_v = (
                [s for s in velocities[i] if random.random() < P_INERTIA] +
                [s for s in v_cog if random.random() < P_COG] +
                [s for s in v_soc if random.random() < P_SOC]
            )
            if len(new_v) > vmax: new_v = random.sample(new_v, vmax)
            velocities[i] = new_v
            swarm[i] = apply_velocity(swarm[i], new_v)
            
    # Pemetaan Hasil ke JSON
    rute_per_kurir_json = []
    for idx, r in enumerate(gbest_routes):
        urutan_nama = [lokasi_list[0]]
        koordinat_list = []
        
        depot_match = coord_df[coord_df['Nama Puskesmas'].str.strip() == lokasi_list[0].strip()]
        if not depot_match.empty:
            koordinat_list.append([
                clean_coordinate(depot_match.iloc[0]['Latitude'], is_lat=True),
                clean_coordinate(depot_match.iloc[0]['Longitude'], is_lat=False)
            ])
        else:
            koordinat_list.append([-7.32229, 112.77177])
            
        for n in r["nodes"]:
            nama_pusk = lokasi_list[n]
            urutan_nama.append(nama_pusk)
            pusk_match = coord_df[coord_df['Nama Puskesmas'].str.strip() == nama_pusk.strip()]
            if not pusk_match.empty:
                koordinat_list.append([
                    clean_coordinate(pusk_match.iloc[0]['Latitude'], is_lat=True),
                    clean_coordinate(pusk_match.iloc[0]['Longitude'], is_lat=False)
                ])
            else:
                koordinat_list.append([0.0, 0.0])
                
        urutan_nama.append(lokasi_list[0])
        koordinat_list.append(koordinat_list[0])
        
        rute_per_kurir_json.append({
            "id_kurir": idx + 1,
            "waktu_tempuh_menit": round(r["waktu"], 2),
            "jarak_tempuh_km": round(r["jarak"], 2),
            "urutan_kunjungan": urutan_nama,
            "koordinat_kunjungan": koordinat_list
        })
        
    return {
        "algoritma": "DPSO",
        "klaster": "",
        "total_kurir": len(rute_per_kurir_json),
        "total_waktu_semua_menit": round(gbest_time, 2),
        "total_jarak_semua_km": round(gbest_dist, 2),
        "riwayat_konvergensi": riwayat_fitness,  # 🌟 STEP 3: Kembalikan list konvergensi ke pemanggil fungsi
        "rute_per_kurir": rute_per_kurir_json
    }

In [21]:
import os
import pandas as pd
import json
import time  # Ditambahkan untuk tracking waktu komputasi secara presisi

# Jalur relatif andalanmu
DATA_DIR = "../data"
OUTPUT_DIR = "../output_json"
path_koordinat = os.path.join(DATA_DIR, "koordinat_eas.csv")

if os.path.exists(path_koordinat):
    # 1. Memuat dataset koordinat puskesmas
    df_coords = pd.read_csv(path_koordinat)

    # 2. Bangun priority dictionary mapping secara otomatis
    priority_map = {}
    for _, row in df_coords.iterrows():
        nama_pusk = str(row['Nama Puskesmas']).strip()
        priority_map[nama_pusk] = get_priority_score(row['Jaringan Pelayanan'], row['Jenis Layanan'])

    # Daftar klaster yang dieksekusi secara berurutan
    daftar_klaster = ["barat", "pusat", "selatan", "timur", "utara"]
    semua_file = os.listdir(DATA_DIR)

    # Struktur master JSON baru (hasil_per_klaster diubah menjadi dictionary)
    output_gabungan = {
        "algoritma": "Discrete Particle Swarm Optimization (DPSO)", # Ubah manual teks ini jika sedang running ga.ipynb
        "grand_total_kurir_surabaya": 0,
        "grand_total_jarak_seluruh_km": 0.0,
        "grand_total_waktu_seluruh_menit": 0.0,
        "grand_total_waktu_komputasi_seluruh_detik": 0.0,
        "hasil_per_klaster": {}
    }

    print("🚀 [START] Memulai optimasi batch dengan metrik Konvergensi & Komputasi...\n")
    print("-" * 75)

    for klaster in daftar_klaster:
        # Cari file jarak & waktu secara fleksibel per klaster
        file_jarak_nama = next((f for f in semua_file if "jarak" in f.lower() and klaster in f.lower() and f.endswith('.csv')), None)
        file_waktu_nama = next((f for f in semua_file if "waktu" in f.lower() and klaster in f.lower() and f.endswith('.csv')), None)

        if file_jarak_nama and file_waktu_nama:
            file_jarak = os.path.join(DATA_DIR, file_jarak_nama)
            file_waktu = os.path.join(DATA_DIR, file_waktu_nama)
            
            print(f"🔄 Processing Klaster: {klaster.upper()}...")
            
            # ⏱️ Mulai hitung waktu komputasi
            start_time = time.time()
            
            # Jalankan mesin algoritma utama sesuai file notebook yang aktif (DPSO / GA)
            hasil_klaster = run_dpso_vrp(file_jarak, file_waktu, df_coords, priority_map)
            
            # ⏱️ Selesai hitung waktu komputasi
            end_time = time.time()
            waktu_detik = round(end_time - start_time, 3)
            
            klaster_key = klaster.capitalize() # Mengubah 'barat' menjadi 'Barat'
            
            # Mapping ke dalam format struktur JSON baru pesananmu
            output_gabungan["hasil_per_klaster"][klaster_key] = {
                "total_kurir": hasil_klaster.get("total_kurir", 0),
                "waktu_komputasi_detik": waktu_detik,
                "total_waktu_semua_menit": hasil_klaster.get("total_waktu_semua_menit", 0.0),
                "total_jarak_semua_km": hasil_klaster.get("total_jarak_semua_km", 0.0),
                "riwayat_konvergensi": hasil_klaster.get("riwayat_konvergensi", []),
                "rute_per_kurir": hasil_klaster.get("rute_per_kurir", [])
            }
            
            # Akumulasi nilai untuk Grand Total global
            output_gabungan["grand_total_kurir_surabaya"] += hasil_klaster.get("total_kurir", 0)
            output_gabungan["grand_total_jarak_seluruh_km"] += hasil_klaster.get("total_jarak_semua_km", 0.0)
            output_gabungan["grand_total_waktu_seluruh_menit"] += hasil_klaster.get("total_waktu_semua_menit", 0.0)
            output_gabungan["grand_total_waktu_komputasi_seluruh_detik"] += waktu_detik
            
            print(f"   ✅ {klaster.upper()} Selesai! [{waktu_detik} detik | {len(hasil_klaster.get('riwayat_konvergensi', []))} Iterasi]\n")
        else:
            print(f"⚠️ Skip Klaster {klaster.upper()}: File tidak lengkap di folder '../data'\n")

    # Rapikan format desimal akhir
    output_gabungan["grand_total_jarak_seluruh_km"] = round(output_gabungan["grand_total_jarak_seluruh_km"], 2)
    output_gabungan["grand_total_waktu_seluruh_menit"] = round(output_gabungan["grand_total_waktu_seluruh_menit"], 2)
    output_gabungan["grand_total_waktu_komputasi_seluruh_detik"] = round(output_gabungan["grand_total_waktu_komputasi_seluruh_detik"], 3)

    # 3. Simpan berkas hasil akhir ke folder ../output_json/
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    file_output_path = os.path.join(OUTPUT_DIR, "rute_dpso.json")
    
    with open(file_output_path, "w") as f:
        json.dump(output_gabungan, indent=4, fp=f)
        
    print("-" * 75)
    print(f"🎉 [SUCCESS] Tugas selesai! Data riwayat konvergensi & komputasi berhasil direkam.")
    print(f"💾 File master JSON baru disimpan di: {file_output_path}")
    print("-" * 75)
else:
    print(f"❌ Gagal! File utama tidak ditemukan di jalur: {path_koordinat}")

🚀 [START] Memulai optimasi batch dengan metrik Konvergensi & Komputasi...

---------------------------------------------------------------------------
🔄 Processing Klaster: BARAT...
   ✅ BARAT Selesai! [0.451 detik | 300 Iterasi]

🔄 Processing Klaster: PUSAT...
   ✅ PUSAT Selesai! [0.127 detik | 160 Iterasi]

🔄 Processing Klaster: SELATAN...
   ✅ SELATAN Selesai! [0.63 detik | 300 Iterasi]

🔄 Processing Klaster: TIMUR...
   ✅ TIMUR Selesai! [0.359 detik | 225 Iterasi]

🔄 Processing Klaster: UTARA...
   ✅ UTARA Selesai! [0.388 detik | 300 Iterasi]

---------------------------------------------------------------------------
🎉 [SUCCESS] Tugas selesai! Data riwayat konvergensi & komputasi berhasil direkam.
💾 File master JSON baru disimpan di: ../output_json/rute_dpso.json
---------------------------------------------------------------------------
